## Tests for `newrelic_sb_sdk.utils.download`

## Imports

In [ ]:
import os
from unittest.mock import MagicMock, patch

import pytest

from newrelic_sb_sdk.utils.download import download_file, download_files

## Tests

In [ ]:
class TestUtilsDownload:
    @patch("requests.get")
    def test_download_file(self, mock_get):
        mock_response = MagicMock()
        mock_response.headers = {"content-length": "10"}
        mock_response.iter_content.return_value = [b"test_chunk"]
        mock_get.return_value = mock_response

        file_name = "test_download.txt"
        download_file(url="http://example.com/test", file_name=file_name)

        mock_get.assert_called_once_with(
            "http://example.com/test", stream=True, timeout=60
        )
        assert os.path.exists(file_name)
        with open(file_name, "rb") as f:
            assert f.read() == b"test_chunk"

        os.remove(file_name)

    @patch("requests.get")
    def test_download_file_no_filename(self, mock_get):
        mock_response = MagicMock()
        mock_response.headers = {"content-length": "10"}
        mock_response.iter_content.return_value = [b"test_chunk"]
        mock_get.return_value = mock_response

        download_file(url="http://example.com/test_auto.txt", file_name="")

        assert os.path.exists("test_auto.txt")
        os.remove("test_auto.txt")

    @patch("requests.get")
    def test_download_file_zero_size(self, mock_get):
        mock_response = MagicMock()
        mock_response.headers = {"content-length": "0"}
        mock_response.iter_content.return_value = []
        mock_get.return_value = mock_response

        file_name = "test_zero.txt"
        with pytest.warns(UserWarning, match="Size of test_zero.txt file is 0B."):
            download_file(url="http://example.com/test", file_name=file_name)

        assert os.path.exists(file_name)
        os.remove(file_name)

    @patch("requests.get")
    def test_download_files(self, mock_get):
        mock_response = MagicMock()
        mock_response.headers = {"content-length": "10"}
        mock_response.iter_content.return_value = [b"test_chunk"]
        mock_get.return_value = mock_response

        urls = ["http://example.com/test1", "http://example.com/test2"]
        base_name = "test_batch"
        ext = "txt"

        download_files(urls=urls, base_file_name=base_name, file_extension=ext)

        assert mock_get.call_count == 2

        file_1 = "test_batch_0.txt"
        file_2 = "test_batch_1.txt"

        assert os.path.exists(file_1)
        assert os.path.exists(file_2)

        os.remove(file_1)
        os.remove(file_2)